In [19]:
#!/Users/ginoprasad/miniconda3/envs/google-api/bin/python3

import os, glob
import sys
import settings
import json
import concurrent.futures
from google.cloud import pubsub_v1
import google.auth
import subprocess as sp
import time
import base64
import re

import utils
from tqdm import tqdm

In [2]:
def extract_text(payload):
    if 'parts' in payload:
        return ''.join(map(extract_text, payload['parts']))
    elif payload.get('mimeType') == 'text/plain':
        data = payload.get('body', {}).get('data', '')
        if data:
            return base64.urlsafe_b64decode(data).decode('utf-8', errors='ignore')
    return ''


list_res = utils.service.users().messages().list(userId='me', q='in:inbox', maxResults=1000).execute()
messages = list_res.get('messages', [])

In [66]:
regex_include = [
    r'(?<=Get Code\r\n\[)https://.*?(?=[ \]])',
    r'(?<=Enter this code to sign in\r\n\r\n)[0-9][0-9][0-9][0-9]',
    '(?<=Yes, This Was Me\r\n\[)https://.*?(?=[\]])',
]

regex_exclude = [
    'Please review who’s using your Netflix account',
    'We’ve updated your account with your new payment info',
    'Your Netflix Household has been confirmed',
    'we’re updating our prices',
    "Here's a quick summary of key updates to reflect our new features and services",
    'We recently announced that',
    'Your 4K upgrade ends soon'
]

emails = [
    'info@account.netflix.com'
]

In [ ]:
for message in tqdm(messages):
    eid = message['id']
    msg = utils.service.users().messages().get(
        userId='me', id=eid, format='full'
    ).execute()

    
    email_from = ''.join([x['value'] for x in msg['payload']['headers'] if x['name'] == 'From'])
    email_from = re.search(r'(?<=<).*(?=>)', email_from).group()
    
    labels = msg.get('labelIds', [])
    if 'INBOX' in labels and 'SENT' not in labels and email_from in emails:
        payload = extract_text(msg['payload'])
                
        for regex in regex_include:
            code = re.search(regex, payload)
            if code is not None:
                code = code.group()
                break
        
        if code is None:
            exclude = False
            for regex in regex_exclude:
                code = re.search(regex, payload)
                if code is not None:
                    exclude = True
                    break
            if exclude:
                continue
            
            with open(settings.payload_path, 'w') as outfile:
                outfile.write(payload)

            print(payload)
            break
        else:
            print(code)

  0%|                          | 1/500 [00:00<01:58,  4.21it/s]

In [55]:
re.search(regex_include[-1], payload)

''

In [56]:
payload

"This link expires in 15 minutes\r\n\r\nDid you request to update your Netflix Household?\r\n\r\nHi Neil,\r\n\r\nWe received a request to update the Netflix Household for\r\nyour account on April 29th, 3:24 PM PDT.\r\n\r\nYour Netflix Household is a collection of your Netflix\r\ndevices that are associated with your internet.\r\n\r\nIf this request was from you or someone who lives with you,\r\ncontinue updating your Netflix Household below.\r\n\r\nIf you did not initiate this request, consider changing your\r\npassword.\r\n[https://www.netflix.com/password?g=4972f652-193f-499b-bab4-831a40bc9802&lkid=URL_PASSWORD&lnktrk=EVO&nftoken=BgjXuOvcAxK4AfLToKXGLq5TXB0UBrRfSPNnw4PhwwAX6JEK59LEdQk%2F%2F0p1Ykc2qyV9xX2x2y9RVyv4kzdHRxRA4d7nYTxXruyd%2Fd9sAPxG1byxlR%2BFKZdCw0G2HEejyPw4JB3%2BCdD2syFnXRMP2MZgPmNDoHceKR7BL80NfJUO7Y1mZroptCnSBcUJqQPTepVEMHKLlh79wVllUXk%2BqYtC0aV88NPMxGJk8Fe0muPr7RfbL37dMYF3kircxqLjR0kYBiIOCgxrdJzWVIinboga%2BhM%3D]\r\n\r\n\xa0\r\n\r\n\xa0\r\n\xa0\r\n\r\nYes, This Was Me\r\

In [26]:
re.search(r'(?<=<).*(?=>)', email_from).group()

'ginoprasad@gmail.com'

In [37]:
msg['snippet']

'Please review who&#39;s using your Netflix account, ginoprasad3@gmail.com. ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏'

In [34]:
code

In [32]:
for regex in regex_list:
    code = re.search(regex, payload)
    if code is not None:
        code = code.group()
        break
print(

None
None


In [15]:





# labels = msg.get('labelIds', [])
# if 'INBOX' in labels and 'SENT' not in labels and 'info@account.netflix.com' in email_from:
#     payload = extract_text(msg['payload'])
#     with open("/Users/ginoprasad/Scripts/EmailManager/test.txt", 'w') as outfile:
#         outfile.write(payload)





        
    # link = re.search(r'(?<=Get Code\r\n\[)https://.*?(?=[ \]])', payload).group()
    # message = f'Netflix Code: {link}'
    # assert len(set("\n'\"") - set(message)) == 3

    # cmd = f"'{settings.send_text_path}' '{settings.group_chat_id}' '{message}'"
    # sp.run(cmd, shell=True)

In [18]:
code

'6646'

In [14]:

code

In [10]:
payload

'Enter this code to sign in\r\n\r\nEnter this code to sign in\r\n\r\n6646\r\n\r\nEnter the code above on your device to sign in to Netflix.\r\nThis code will expire in 15 minutes.\r\n\r\nIf you didn’t send this request, you can ignore this email\r\nor review your recent device activity.\r\n[https://www.netflix.com/accountaccess?g=5c03f486-1f81-4951-bb10-eb7d59d9bac1&lkid=URL_ACCOUNT_ACCESS&lnktrk=EVO]\r\n\r\nTo help security, don’t share this code with anyone outside\r\nyour household.\r\n\r\nThe Netflix team\r\n\r\n\xa0\r\n\xa0\r\n\r\n   Questions? Call 1-844-569-7700\r\n   \r\n   121 Albright Way, Los Gatos, CA 95032, U.S.A.\r\n   [https://help.netflix.com/legal/corpinfo?g=5c03f486-1f81-4951-bb10-eb7d59d9bac1&lkid=URL_CORP_INFO&lnktrk=EVO]\r\n   \r\n   Terms of Use\r\n   [https://www.netflix.com/TermsOfUse?g=5c03f486-1f81-4951-bb10-eb7d59d9bac1&lkid=URL_TERMS&lnktrk=EVO]\r\n   Privacy\r\n   [https://www.netflix.com/PrivacyPolicy?g=5c03f486-1f81-4951-bb10-eb7d59d9bac1&lkid=URL_PRIVACY

In [ ]:
link = .group()

In [ ]:


6646